# JN0b · What a computational notebook is

*On-ramp 2 of 8.*

A city PDF says **708 homes were completed in 2024**. How do you *check* that number instead of just trusting it? A printed report is a claim you can't open. The tool you're reading right now — a **notebook** — is a claim you can.

### Running the cells

To run a cell, click it and press **Shift + Return**, or click the **run (▸) button** on the cell. The simplest way through any notebook here is to start at the top and run each cell in order, reading the output that appears beneath it.

Some of the computational cells may look complex right now — that's expected, and it's fine. **You don't need to understand every line yet;** the ideas become clear as you go. Run them, watch what they produce, and keep moving.

## (run first) Colab setup

Fetches the data + shared modules from R2. **No-op if you already have the repo locally.** On Colab it recreates the minimal layout so the cells below find everything.

In [1]:
# === COLAB BOOTSTRAP - fetch curriculum data + modules from R2 (NO-OP if the repo is local) ===
from pathlib import Path
import sys, urllib.request, urllib.parse, tarfile, subprocess

R2 = 'https://pub-2cee87f70da64080ab70ee0a34b55099.r2.dev/curriculum'
USE_CLEAN = False   # False: raw .xlsx path (JN1's messy-data lesson).  True (skip-ingest): permits_clean.*

_here = Path.cwd()
_have_repo = (_here/'scripts'/'build_v2').exists() or any((p/'scripts'/'build_v2').exists() for p in _here.parents)

def _get(url):
    # r2.dev sits behind Cloudflare, which 403s the default 'Python-urllib' User-Agent; send a browser UA.
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=60) as r:
        return r.read()

if _have_repo:
    print('local repo detected - no fetch needed')
else:
    try:
        import pyarrow  # the parquet / USE_CLEAN path needs it; Colab has pandas, maybe not pyarrow
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow'], check=True)
    def _fetch(url, dest):
        dest = Path(dest)
        if dest.exists():
            return                                   # cached: re-runs don't re-download
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(_get(url)); print('fetched', dest.name)
    # 1) shared modules -> ./scripts/...  (the config-cell repo-root walk then finds scripts/build_v2)
    if not (_here/'scripts'/'build_v2').exists():
        Path('modules.tgz').write_bytes(_get(f'{R2}/curriculum_modules.tar.gz'))
        _tar = tarfile.open('modules.tgz')
        try: _tar.extractall(_here, filter='data')      # py3.12+: safe extract, no deprecation warning
        except TypeError: _tar.extractall(_here)         # older python has no filter arg
        _tar.close(); Path('modules.tgz').unlink(missing_ok=True)   # tidy: drop the intermediate tarball
        print('extracted modules -> ./scripts/')
    # 2) data -> the SAME relative paths the notebooks use (raw .xlsx AND clean exports, both fetched)
    for rel in ['data/raw/cpra-downloads/BP_Annual Permit Report-2018-2022.xlsx',
                'data/raw/cpra-downloads/BP_Annual Permit Report-2023-2025.xlsx',
                'databases/hcd_apr_mirror_2026-06-17_fresh.db',
                'databases/hcd_apr_mirror.db',
                'data/processed/permits_clean.csv',
                'data/processed/permits_clean.parquet',
                'data/processed/permits_clean_README.md']:
        _fetch(f"{R2}/data/{urllib.parse.quote(rel.split('/')[-1])}", _here/rel)   # quote -> %20 for the spaced .xlsx names
    print('curriculum bundle ready (fetched from R2)')


local repo detected - no fetch needed


In [2]:
def md(t):
    from IPython.display import Markdown, display
    display(Markdown(t))

## Point the notebook at the data

Finds the repo root, locates the permit feed, and puts the project's real shared code on the path. The two knobs near the top are all a student changes to run this on another city.

In [3]:
# === CONFIG — point this at YOUR city's permit data (this notebook is clonable) ===
from pathlib import Path
import sys, glob

# walk up to the repo root (where scripts/build_v2 lives) so the notebook runs from anywhere
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'scripts' / 'build_v2').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

# --- the two knobs a student changes for another city ---
PERMIT_GLOB   = str(REPO_ROOT / 'data/raw/cpra-downloads/BP_Annual Permit Report-*.xlsx')
HEADER_ROW    = 7        # 0-indexed: Berkeley's CPRA export puts the column names on row 8
EXPECTED_UNIQUE = 30764  # the known unique-permit total for YOUR feed (Berkeley = 30,764)

# import the REAL shared modules the pipeline uses (we demonstrate them, never reinvent)
sys.path.insert(0, str(REPO_ROOT / 'scripts'))
sys.path.insert(0, str(REPO_ROOT / 'scripts' / 'build_v2'))
print('repo root :', REPO_ROOT)
print('feed files:', [Path(f).name for f in glob.glob(PERMIT_GLOB)])


repo root : /Users/johngage/berkeley-data
feed files: ['BP_Annual Permit Report-2023-2025.xlsx', 'BP_Annual Permit Report-2018-2022.xlsx']


## A document that runs

A **notebook** mixes two kinds of **cells**: *markdown* cells (prose, like this one) and *code* cells (which the computer runs). Behind it sits a **kernel** — a live Python session that remembers what earlier cells did. You **run** a cell to produce its **output**; you can **re-run** it any time. **Colab** is Google's free website for running notebooks in a browser, which is why this course opens with a one-time *setup* cell.

In [4]:
years = 2025 - 2018                       # the span the feed covers, computed (not typed) from its end years
print('the permit feed spans', years, 'years')
# Try it: change 2018 to 2020, re-run — the output updates. Nothing here is frozen.

the permit feed spans 7 years


## Why this beats a report: re-running is re-checking

In a PDF, *708* is just ink. In a notebook, every number is produced by code you can read and run again. Let's prove it by deriving a number straight from the source data — a **checkpoint** habit we'll lean on the whole way: *state a number, then make the computer reproduce it.*

In [5]:
import pandas as pd, glob
def _load(p):
    # read one spreadsheet at its real header row, then tidy the column names
    d = pd.read_excel(p, dtype=str, header=HEADER_ROW); d.columns = [str(c).strip() for c in d.columns]; return d
df = pd.concat([_load(f) for f in glob.glob(PERMIT_GLOB)], ignore_index=True)   # stack every yearly file into one table
df = df[df['PermitNumber'].notna()].copy()        # drop rows that have no permit number
n = df['PermitNumber'].nunique()                  # count the distinct permit numbers
md(f'''There are **{n:,}** distinct permits in the feed — a number you just *re-derived from the source*, not one copied from a report. Re-run this cell and it says the same thing, because it recomputes every time. **That is the point of a notebook: every number carries its own proof, and re-running is re-checking.**''')

There are **30,764** distinct permits in the feed — a number you just *re-derived from the source*, not one copied from a report. Re-run this cell and it says the same thing, because it recomputes every time. **That is the point of a notebook: every number carries its own proof, and re-running is re-checking.**

In [6]:
# pull the year off the front of each permit number
yr = df['PermitNumber'].str.extract(r'^[A-Za-z]+(\d{4})')[0]
n2024 = int((yr == '2024').sum()); n2023 = int((yr == '2023').sum())   # how many permits carry each year
md(f'''Now a narrower question — *how many permits carry a 2024 number?* → **{n2024:,}**. Switch the year to 2023 and it becomes **{n2023:,}**. Notice this cell used the `df` the previous cell built: cells form a **chain**, each standing on the one above, and changing one input ripples downstream. A notebook is a sequence of living steps, not a frozen page.''')

Now a narrower question — *how many permits carry a 2024 number?* → **3,890**. Switch the year to 2023 and it becomes **4,124**. Notice this cell used the `df` the previous cell built: cells form a **chain**, each standing on the one above, and changing one input ripples downstream. A notebook is a sequence of living steps, not a frozen page.

## Where the notebook came from

The idea that code and explanation should live in one runnable document has a real lineage. Donald **Knuth** named it *literate programming* in 1984 — prose and code woven together. The first widely used *interactive, re-runnable* form was the **Mathematica notebook**, which Stephen Wolfram created and shipped as **Mathematica 1.0 on June 23, 1988** (its notebook front end designed by Theodore Gray). **Jupyter** — the tool this course runs in — is the open-source descendant created by Fernando Pérez that brought the notebook to Python, free for anyone. A thirty-five-year-old idea, in the form you can pick up today for nothing.

**Next — JN0c:** the smallest reusable piece of code — the **function** — which asks the same question 30,000 times, identically.